In [15]:
import joblib
import pandas as pd
import os


pipeline = joblib.load("modele_faux_billets.joblib")

In [16]:
print(pipeline)

Pipeline(steps=[('imputer',
                 IterativeImputer(estimator=LinearRegression(), random_state=42,
                                  skip_complete=True)),
                ('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(C=0.1, max_iter=1000, penalty='l1',
                                    solver='liblinear'))])


In [18]:
import joblib
import pandas as pd
import os


pipeline = joblib.load("modele_faux_billets.joblib")

DATA_PATH = "billets.csv"

if not os.path.exists(DATA_PATH):
    print(f"❌ Attention : le fichier '{DATA_PATH}' n'existe pas.")

try:
    df = pd.read_csv(DATA_PATH, sep=";")
    print("✅ Données chargées :", df.shape)
except Exception as e:
    print("❌ Erreur lors du chargement :", e)
    print("⚠️ Tu as bien lu le code avant de l'executer ? 😏")

# On ne garde que les colonnes nécessaires pour la prédiction (diagonal	height_left	height_right	margin_low	margin_up	length)
df = df[["diagonal", "height_left", "height_right", "margin_low", "margin_up", "length"]]

########################################

# Faire les if pour afficher les nan et les imputer

#######################################

df = pipeline['imputer'].transform(df)
#Transformer le tableau numpy en DataFrame pour garder les noms de colonnes
df = pd.DataFrame(df, columns=["diagonal", "height_left", "height_right", "margin_low", "margin_up", "length"])
print("Imputatipon des valeurs manquantes")

prediction = pipeline[1:].predict(df)
proba = pipeline.predict_proba(df)

# Ajout de la prediction dans le DataFrame (If 1 = Vrai billet, 0 = Faux billet)
df["Prédiction"] = prediction
df["Résultat"] = ["Faux billet" if p == 1 else "Vrai billet" for p in prediction]

# Ajout de la probabilité de true et de false dans le DataFrame
df["Probabilité d'un faux billet"] = (proba[:, 1]*100).round(2)
df["Probabilité d'un vrai billet"] = (proba[:, 0]*100).round(2)

#Display les lignes Vrai billet
display(df[df["Résultat"] == "Faux billet"].head(30))

# Print du nombre de faux et de vrais billets
print(f"Nombre de vrais billets : {(prediction == 0).sum()}")
print(f"Nombre de faux billets : {(prediction == 1).sum()}")


✅ Données chargées : (1500, 7)
Imputatipon des valeurs manquantes


c:\Users\flepineux\AppData\Local\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


,diagonal,height_left,height_right,margin_low,margin_up,length,Prédiction,Résultat,Probabilité d'un faux billet,Probabilité d'un vrai billet
591,171.67,103.81,103.76,4.590000,3.30,112.18,1,Faux billet,74.88,25.12
728,171.94,104.11,104.16,4.080000,3.35,111.76,1,Faux billet,82.47,17.53
946,171.63,103.87,104.66,4.700537,3.27,112.68,1,Faux billet,62.49,37.51
1000,172.28,103.95,103.91,4.780000,3.31,111.40,1,Faux billet,98.75,1.25
1001,171.92,103.86,104.30,4.960000,3.13,111.29,1,Faux billet,99.15,0.85
1002,171.59,104.14,104.38,4.970000,3.47,111.22,1,Faux billet,99.88,0.12
1003,172.02,104.33,104.33,5.190000,3.21,111.99,1,Faux billet,98.07,1.93
1004,172.55,104.25,104.23,5.600000,3.13,111.72,1,Faux billet,99.62,0.38
1005,171.88,104.30,104.18,5.340000,3.33,112.69,1,Faux billet,92.97,7.03
1006,171.63,104.05,104.25,4.610000,3.10,110.91,1,Faux billet,99.19,0.81


Nombre de vrais billets : 1006
Nombre de faux billets : 494


In [19]:
# Print du nombre de lignes du DataFrame avec prediction = 1
print(f"Nombre de lignes avec prediction = 1 : {df[df['Prédiction'] == 1].shape[0]}")

Nombre de lignes avec prediction = 1 : 494


In [20]:
df_verif = pd.read_csv(DATA_PATH, sep=";")

In [21]:
# Merge de df et df_verif sur l'index pour comparer les résultats 
df_merged = df.merge(df_verif['is_genuine'], left_index=True, right_index=True, suffixes=('_pred', '_verif'))
df_merged["is_fake"] = df_merged["is_genuine"].apply(lambda x: 0 if x == 1 else 1)
df_merged = df_merged.drop(columns=["is_genuine"])
df_merged["Résultat correct"] = df_merged.apply(lambda row: "Correct" if row["Prédiction"] == row["is_fake"] else "Incorrect", axis=1)

In [22]:
# Print du nombre de lignes du DataFrame correct et incorrect
print(f"Nombre de lignes correctes : {df_merged[df_merged['Résultat correct'] == 'Correct'].shape[0]}")
print(f"Nombre de lignes incorrectes : {df_merged[df_merged['Résultat correct'] == 'Incorrect'].shape[0]}")

Nombre de lignes correctes : 1488
Nombre de lignes incorrectes : 12


In [23]:
#Affichage des lignes Incorrectes
display(df_merged[df_merged["Résultat correct"] == "Incorrect"].head(30))

,diagonal,height_left,height_right,margin_low,margin_up,length,Prédiction,Résultat,Probabilité d'un faux billet,Probabilité d'un vrai billet,is_fake,Résultat correct
591,171.67,103.81,103.76,4.590000,3.30,112.18,1,Faux billet,74.88,25.12,0,Incorrect
728,171.94,104.11,104.16,4.080000,3.35,111.76,1,Faux billet,82.47,17.53,0,Incorrect
946,171.63,103.87,104.66,4.700537,3.27,112.68,1,Faux billet,62.49,37.51,0,Incorrect
1025,172.17,104.20,104.13,3.860000,3.38,112.44,0,Vrai billet,24.97,75.03,1,Incorrect
1073,172.13,103.67,103.82,4.270000,3.22,112.15,0,Vrai billet,45.39,54.61,1,Incorrect
1083,171.85,103.60,103.82,4.600000,3.21,112.50,0,Vrai billet,41.50,58.50,1,Incorrect
1103,171.88,104.05,103.75,4.410000,3.21,112.52,0,Vrai billet,31.35,68.65,1,Incorrect
1122,172.09,104.15,104.17,4.150000,3.40,113.85,0,Vrai billet,1.07,98.93,1,Incorrect
1160,172.39,104.05,104.32,4.130000,3.41,112.66,0,Vrai billet,31.94,68.06,1,Incorrect
1190,171.45,104.21,104.18,4.550000,3.52,113.21,0,Vrai billet,32.75,67.25,1,Incorrect


In [24]:
display(df_merged.head(30))

,diagonal,height_left,height_right,margin_low,margin_up,length,Prédiction,Résultat,Probabilité d'un faux billet,Probabilité d'un vrai billet,is_fake,Résultat correct
0,171.81,104.86,104.95,4.52,2.89,112.83,0,Vrai billet,20.77,79.23,0,Correct
1,171.46,103.36,103.66,3.77,2.99,113.09,0,Vrai billet,0.25,99.75,0,Correct
2,172.69,104.48,103.50,4.40,2.94,113.16,0,Vrai billet,1.92,98.08,0,Correct
3,171.36,103.91,103.94,3.62,3.01,113.51,0,Vrai billet,0.08,99.92,0,Correct
4,171.73,104.28,103.46,4.04,3.48,112.54,0,Vrai billet,28.47,71.53,0,Correct
5,172.17,103.74,104.08,4.42,2.95,112.81,0,Vrai billet,6.29,93.71,0,Correct
6,172.34,104.18,103.85,4.58,3.26,112.81,0,Vrai billet,31.23,68.77,0,Correct
7,171.88,103.76,104.08,3.98,2.92,113.08,0,Vrai billet,0.65,99.35,0,Correct
8,172.47,103.92,103.67,4.00,3.25,112.85,0,Vrai billet,4.50,95.50,0,Correct
9,172.47,104.07,104.02,4.04,3.25,113.45,0,Vrai billet,1.17,98.83,0,Correct
